In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import wandb
import torchaudio.transforms as T
from sklearn.metrics import f1_score

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

2026-04-04 00:59:12.685900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775264352.906154      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775264352.972441      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775264353.507999      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775264353.508044      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775264353.508046      24 computation_placer.cc:177] computation placer alr

In [3]:
DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
BASE_PATH = DATA_PATH   

SAMPLE_RATE = 16000
CROP_DURATION = 30
SNR_MIN = 5
SNR_MAX = 20

STEMS = ["drums", "vocals", "bass", "other"]

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

GENRE_TO_IDX = {genre: idx for idx, genre in enumerate(GENRES)}
IDX_TO_GENRE = {idx: genre for genre, idx in GENRE_TO_IDX.items()}


BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 3e-5
NUM_WORKERS = 4


SAMPLES_PER_GENRE_TRAIN = 150
SAMPLES_PER_GENRE_VAL = 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
def load_audio(filepath, sr=SAMPLE_RATE):
    
    try:
        audio, _ = librosa.load(filepath, sr=sr, mono=True)
        return audio
    except Exception as e:
        print(f"Error loading: {filepath}")
        return None

def crop_audio(audio, duration_sec, sr=SAMPLE_RATE):
    
    target_length = duration_sec * sr
   
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        start = random.randint(0, len(audio) - target_length)
        audio = audio[start:start + target_length]
   
    return audio
    
def add_noise(audio, noise, snr_db):
    
    if len(noise) < len(audio):
        noise = np.tile(noise, int(np.ceil(len(audio)/len(noise))))
    noise = noise[:len(audio)]
   
    
    start = random.randint(0, len(audio)//2)
    end = min(len(audio), start + len(noise)//2)
   
    segment = audio[start:end]
    noise_segment = noise[:end-start]
   
    signal_power = np.mean(segment ** 2)
    noise_power = np.mean(noise_segment ** 2)
   
    if noise_power > 0:
        snr_linear = 10 ** (snr_db / 10)
        scale = np.sqrt(signal_power / (noise_power * snr_linear))
        audio[start:end] += noise_segment * scale
   
    
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    return audio

def get_noise_paths(data_path):
    """Get all noise file paths"""
    noise_path = Path(data_path) / "ESC-50-master" / "audio"
    if not noise_path.exists():
        return []
    return [str(f) for f in noise_path.glob("*.wav")]



In [5]:
class OptimizedMashupDataset(Dataset):
    
   
    def __init__(self, data_path, genres, stems, noise_paths,
                 num_samples_per_genre=150, sr=SAMPLE_RATE,
                 crop_duration=CROP_DURATION):
       
        self.data_path = Path(data_path)
        self.genres = genres
        self.stems = stems
        self.noise_paths = noise_paths
        self.num_samples_per_genre = num_samples_per_genre
        self.sr = sr
        self.crop_duration = crop_duration
       
        self.stem_dict = self._build_stem_dict()
        self.total_samples = len(genres) * num_samples_per_genre
        
    def _build_stem_dict(self):
        stem_dict = {}
        for genre in self.genres:
            genre_path = self.data_path / "genres_stems" / genre
            stem_dict[genre] = {stem: [] for stem in self.stems}
           
            if not genre_path.exists():
                continue
               
            for song in os.listdir(genre_path):
                song_path = genre_path / song
                if not song_path.is_dir():
                    continue
               
                for stem in self.stems:
                    file = song_path / f"{stem}.wav"
                    if file.exists():
                        stem_dict[genre][stem].append(str(file))
       
        return stem_dict

    def _generate_mashup(self, genre):
        
        selected_paths = []
        for stem in self.stems:
            if self.stem_dict[genre][stem]:
                selected_paths.append(random.choice(self.stem_dict[genre][stem]))
       
        if len(selected_paths) < 2:
            return None
       
       
        audios = []
        for path in selected_paths:
            y = load_audio(path, self.sr)
            if y is not None:
                audios.append(y)
       
        if len(audios) < 2:
            return None
       
        synced = audios   
        
        
        min_len = min(len(y) for y in synced)
        mix = np.zeros(min_len)
        
        for y in synced:
            weight = random.uniform(0.6, 1.4)
            mix += weight * y[:min_len]
       
        mix = mix / (np.max(np.abs(mix)) + 1e-8)
       
        
        mix = crop_audio(mix, self.crop_duration, self.sr)
       
        
        if self.noise_paths and random.random() > 0.2:
            noise = load_audio(random.choice(self.noise_paths), self.sr)
            if noise is not None:
                noise = crop_audio(noise, self.crop_duration, self.sr)
                snr = random.uniform(SNR_MIN, SNR_MAX)
                mix = add_noise(mix, noise, snr)
       
        return mix

    def __len__(self):
        return self.total_samples
   
    def __getitem__(self, idx):
        
       
        genre_idx = idx // self.num_samples_per_genre
        genre = self.genres[genre_idx]
       
        audio = None
        max_attempts = 3
        for _ in range(max_attempts):
            audio = self._generate_mashup(genre)
            if audio is not None:
                break
       
        if audio is None:
            audio = np.zeros(self.crop_duration * self.sr)
       
        label = GENRE_TO_IDX[genre]
       
        return audio, label

In [6]:
class ASTDataset(Dataset):
    
   
    def __init__(self, base_dataset, feature_extractor):
        self.base_dataset = base_dataset
        self.feature_extractor = feature_extractor
   
    def __len__(self):
        return len(self.base_dataset)
   
    def __getitem__(self, idx):
        audio, label = self.base_dataset[idx]
       
        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )
       
        return inputs.input_values.squeeze(0), label

In [7]:

def train_ast_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
   
    pbar = tqdm(dataloader, desc="Training AST")
   
    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)
       
        optimizer.zero_grad()
       
        outputs = model(inputs, labels=labels)
        loss = outputs.loss
       
        loss.backward()
        optimizer.step()
        scheduler.step()
       
        running_loss += loss.item()
       
        logits = outputs.logits
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
       
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
   
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
   
    return epoch_loss, epoch_acc


def validate_ast(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
   
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating AST"):
            inputs = inputs.to(device)
            labels = labels.to(device)
           
            outputs = model(inputs, labels=labels)
            loss = outputs.loss
           
            running_loss += loss.item()
           
            logits = outputs.logits
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
   
    val_loss = running_loss / len(dataloader)
    val_acc = 100. * correct / total
   
    return val_loss, val_acc


# TRAINING

print("\n" + "="*60)
print("OPTIMIZED AST TRAINING ")
print("="*60)

os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="ast-optimized-fast",
    config={
        "model": "AST",
        "pretrained": True,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "samples_per_genre": SAMPLES_PER_GENRE_TRAIN,
        "num_workers": NUM_WORKERS,
        "optimizations": "no_tempo_sync, reduced_samples, larger_batch"
    }
)


noise_paths = get_noise_paths(DATA_PATH)
print(f"Found {len(noise_paths)} noise files")


print("\nCreating datasets...")
train_base = OptimizedMashupDataset(
    DATA_PATH, GENRES, STEMS, noise_paths,
    num_samples_per_genre=SAMPLES_PER_GENRE_TRAIN
)

val_base = OptimizedMashupDataset(
    DATA_PATH, GENRES, STEMS, noise_paths,
    num_samples_per_genre=SAMPLES_PER_GENRE_VAL
)


print("Loading pretrained AST model...")
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

ast_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)


train_ast_dataset = ASTDataset(train_base, feature_extractor)
val_ast_dataset = ASTDataset(val_base, feature_extractor)

train_loader = DataLoader(
    train_ast_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ast_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train samples: {len(train_ast_dataset)}, Val samples: {len(val_ast_dataset)}")
print(f"Steps per epoch: {len(train_loader)}")


optimizer = torch.optim.AdamW(ast_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1
)

# Training loop
print("\nStarting training...")
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
   
    train_loss, train_acc = train_ast_epoch(ast_model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_acc = validate_ast(ast_model, val_loader, DEVICE)
   
    print(f"\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
   
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
   
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ast_model.save_pretrained("ast_best")
        print(f"✓ Saved best AST model (val_acc: {val_acc:.2f}%)")

wandb.finish()

print(f"\n{'='*60}")
print(f"✓ AST Training Complete!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"{'='*60}")


OPTIMIZED AST TRAINING 


wandb: Currently logged in as: 24f1002246 (24f1002246-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run toy6ruq1
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260404_005929-toy6ruq1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ast-optimized-fast
wandb: ⭐️ View project at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: 🚀 View run at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/toy6ruq1


Found 2000 noise files

Creating datasets...
Loading pretrained AST model...


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Exception ignored in: Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00><function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
Traceback (most recent call last):


  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
Traceback (most recent call last):
Traceback (most recent call last):
    self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
      File "/usr/local/lib/python3.12/dist-packages/torch/u

Validating AST:   0%|          | 0/19 [00:00<?, ?it/s]


Train Loss: 0.0527, Train Acc: 98.33%
Val Loss: 0.0523, Val Acc: 98.00%


wandb: 
wandb: Run history:
wandb:         epoch ▁▂▃▃▄▅▆▆▇█
wandb: learning_rate ██▇▆▅▄▃▂▁▁
wandb:     train_acc ▁▇▇▇██████
wandb:    train_loss █▃▂▂▂▁▁▁▁▁
wandb:       val_acc ▁▄▄▆▆▇▇██▇
wandb:      val_loss █▄▄▃▂▂▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 10
wandb: learning_rate 0.0
wandb:     train_acc 98.33333
wandb:    train_loss 0.05269
wandb:       val_acc 98
wandb:      val_loss 0.05227
wandb: 
wandb: 🚀 View run ast-optimized-fast at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/toy6ruq1
wandb: ⭐️ View project at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260404_005929-toy6ruq1/logs



✓ AST Training Complete!
Best Validation Accuracy: 99.33%


In [8]:
print("\n" + "="*60)
print("STARTING INFERENCE WITH TTA")
print("="*60)

def get_multiple_crops(audio, duration_sec, sr=SAMPLE_RATE, num_crops=5):
    """Get multiple crops from audio for TTA"""
    target_length = duration_sec * sr
   
    if len(audio) < target_length:
        return [np.pad(audio, (0, target_length - len(audio)))]
   
    crops = []
    max_start = len(audio) - target_length
   
    if num_crops == 1:
        crops.append(audio[:target_length])
    else:
        step = max_start // (num_crops - 1) if num_crops > 1 else 0
       
        for i in range(num_crops):
            start = min(i * step, max_start)
            crop = audio[start:start + target_length]
            crops.append(crop)
   
    return crops
    
def predict_with_tta(audio, model, feature_extractor, num_crops=5):
    """Predict with Test-Time Augmentation"""
    crops = get_multiple_crops(audio, CROP_DURATION, SAMPLE_RATE, num_crops=num_crops)
   
    all_probs = []
   
    for crop in crops:
        inputs = feature_extractor(
            crop,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )
       
        inputs = inputs.input_values.to(DEVICE)
       
        with torch.no_grad():
            outputs = model(inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy()[0])
   
    
    avg_probs = np.mean(all_probs, axis=0)
    predicted_idx = np.argmax(avg_probs)
    predicted_genre = IDX_TO_GENRE[predicted_idx]
   
    return predicted_genre, avg_probs


STARTING INFERENCE WITH TTA


In [9]:
print("Loading best model for inference...")
ast_model = ASTForAudioClassification.from_pretrained("ast_best").to(DEVICE)
ast_model.eval()


test_df = pd.read_csv(DATA_PATH + "/test.csv")

print(f"Found {len(test_df)} test files")
print("Using TTA with 5 crops per audio")

results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Making predictions"):
   
    file_id = row["id"]
    file_path = DATA_PATH + "/" + row["filename"]
   
    audio = load_audio(file_path, SAMPLE_RATE)
   
    if audio is None:
        predicted_genre = "pop"  # Fallback
    else:
        predicted_genre, probs = predict_with_tta(
            audio, ast_model, feature_extractor, num_crops=5
        )
   
    results.append({
        "id": file_id,
        "genre": predicted_genre
    })

# ============================
# CREATE SUBMISSION
# ============================
print("\n" + "="*60)
print("CREATING SUBMISSION FILE")
print("="*60)

submission_df = pd.DataFrame(results)
submission_df = submission_df.sort_values("id").reset_index(drop=True)
submission_df.to_csv("submission.csv", index=False)

print(f"\n✓ Submission file created: submission.csv")
print(f"Total predictions: {len(submission_df)}")

print("\nFirst 10 predictions:")
print(submission_df.head(10))



Loading best model for inference...
Found 3020 test files
Using TTA with 5 crops per audio


Making predictions:   0%|          | 0/3020 [00:00<?, ?it/s]


CREATING SUBMISSION FILE

✓ Submission file created: submission.csv
Total predictions: 3020

First 10 predictions:
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9        pop
9  10      disco


# Model 2

In [10]:
data_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
sample_rate = 16000
crop_duration = 30
snr_min = 5
snr_max = 20

n_mels = 64
n_fft = 1024
hop_length = 512

stems = ["drums", "vocals", "bass", "other"]
genres = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

genre_to_idx = {genre: idx for idx, genre in enumerate(genres)}
idx_to_genre = {idx: genre for genre, idx in genre_to_idx.items()}
num_classes = 10

batch_size = 32
num_epochs = 8
learning_rate = 2e-3
num_workers = 4

samples_per_genre_train = 150
samples_per_genre_val = 30

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [11]:
def load_audio(file_path, sr=sample_rate):
    try:
        audio, _ = librosa.load(file_path, sr=sr, mono=True)
        return audio
    except Exception as e:
        print(f"Error loading: {file_path}")
        return None

def crop_audio(audio, duration_sec, sr=sample_rate):
    target_length = duration_sec * sr

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        start = random.randint(0, len(audio) - target_length)
        audio = audio[start:start + target_length]

    return audio

def add_noise(audio, noise, snr_db):
    if len(noise) < len(audio):
        noise = np.tile(noise, int(np.ceil(len(audio) / len(noise))))
    noise = noise[:len(audio)]

    start = random.randint(0, len(audio) // 2)
    end = min(len(audio), start + len(noise) // 2)

    segment = audio[start:end]
    noise_segment = noise[:end - start]

    signal_power = np.mean(segment ** 2)
    noise_power = np.mean(noise_segment ** 2)

    if noise_power > 0:
        snr_linear = 10 ** (snr_db / 10)
        scale = np.sqrt(signal_power / (noise_power * snr_linear))
        audio[start:end] += noise_segment * scale

    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    return audio

def get_noise_paths(data_path):
    noise_dir = Path(data_path) / "ESC-50-master" / "audio"
    if not noise_dir.exists():
        return []
    return [str(file) for file in noise_dir.glob("*.wav")]

In [12]:
class OptimizedMashupDataset(Dataset):

    def __init__(self, data_path, genres, stems, noise_paths,
                 num_samples_per_genre=150, sr=sample_rate,
                 crop_duration=crop_duration):

        self.data_path = Path(data_path)
        self.genres = genres
        self.stems = stems
        self.noise_paths = noise_paths
        self.num_samples_per_genre = num_samples_per_genre
        self.sr = sr
        self.crop_duration = crop_duration

        self.stem_dict = self._build_stem_dict()
        self.total_samples = len(genres) * num_samples_per_genre

    def _build_stem_dict(self):
        stem_dict = {}
        for genre in self.genres:
            genre_path = self.data_path / "genres_stems" / genre
            stem_dict[genre] = {stem: [] for stem in self.stems}

            if not genre_path.exists():
                continue

            for song in os.listdir(genre_path):
                song_path = genre_path / song
                if not song_path.is_dir():
                    continue

                for stem in self.stems:
                    file_path = song_path / f"{stem}.wav"
                    if file_path.exists():
                        stem_dict[genre][stem].append(str(file_path))

        return stem_dict

    def _generate_mashup(self, genre):

        selected_paths = []
        for stem in self.stems:
            if self.stem_dict[genre][stem]:
                selected_paths.append(random.choice(self.stem_dict[genre][stem]))

        if len(selected_paths) < 2:
            return None

        audios = []
        for file_path in selected_paths:
            audio = load_audio(file_path, self.sr)
            if audio is not None:
                audios.append(audio)

        if len(audios) < 2:
            return None

        synced_audio = audios

        min_len = min(len(audio) for audio in synced_audio)
        mixed_audio = np.zeros(min_len)

        for audio in synced_audio:
            weight = random.uniform(0.6, 1.4)
            mixed_audio += weight * audio[:min_len]

        mixed_audio = mixed_audio / (np.max(np.abs(mixed_audio)) + 1e-8)

        mixed_audio = crop_audio(mixed_audio, self.crop_duration, self.sr)

        if self.noise_paths and random.random() > 0.2:
            noise = load_audio(random.choice(self.noise_paths), self.sr)
            if noise is not None:
                noise = crop_audio(noise, self.crop_duration, self.sr)
                snr = random.uniform(snr_min, snr_max)
                mixed_audio = add_noise(mixed_audio, noise, snr)

        return mixed_audio

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):

        genre_idx = idx // self.num_samples_per_genre
        genre = self.genres[genre_idx]

        audio = None
        max_attempts = 3
        for _ in range(max_attempts):
            audio = self._generate_mashup(genre)
            if audio is not None:
                break

        if audio is None:
            audio = np.zeros(self.crop_duration * self.sr)

        label = genre_to_idx[genre]

        return audio, label

In [13]:
class CNNDataset(Dataset):

    def __init__(self, base_dataset, augment=False):
        self.base_dataset = base_dataset
        self.augment = augment

        if augment:
            self.freq_mask = T.FrequencyMasking(freq_mask_param=8)
            self.time_mask = T.TimeMasking(time_mask_param=16)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        audio, label = self.base_dataset[idx]

        mel_spec = librosa.feature.melspectrogram(
            y=audio,
            sr=sample_rate,
            n_mels=n_mels,
            n_fft=n_fft,
            hop_length=hop_length
        )

        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)

        mel_spec_tensor = torch.FloatTensor(mel_spec_db).unsqueeze(0)

        if self.augment:
            mel_spec_tensor = self.freq_mask(mel_spec_tensor)
            mel_spec_tensor = self.time_mask(mel_spec_tensor)

        return mel_spec_tensor, label

In [14]:
class LightCNN(nn.Module):

    def __init__(self, num_classes=num_classes):
        super(LightCNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout2d(0.1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout2 = nn.Dropout2d(0.1)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.dropout3 = nn.Dropout2d(0.2)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(128, 64)
        self.dropout_fc = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout1(x)

        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout2(x)

        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout3(x)

        x = self.gap(x)
        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x

In [15]:
def train_cnn_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(dataloader, desc="Training CNN")

    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        acc = 100. * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    epoch_acc = 100. * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

    return epoch_loss, epoch_f1, epoch_acc


def validate_cnn(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating CNN"):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / len(dataloader)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_acc = 100. * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

    return val_loss, val_f1, val_acc

In [16]:
print("\n" + "="*60)

os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="cnn-from-scratch-optimized",
    config={
        "model": "CNN",
        "architecture": "LightCNN",
        "batch_size": batch_size,
        "epochs": num_epochs,
        "learning_rate": learning_rate,
        "n_mels": n_mels,
        "samples_per_genre": samples_per_genre_train,
        "num_workers": num_workers
    }
)

wandb: setting up run 0u8z9x96
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260404_024027-0u8z9x96
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run cnn-from-scratch-optimized
wandb: ⭐️ View project at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: 🚀 View run at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/0u8z9x96


In [17]:
noise_paths = get_noise_paths(data_path)
print(f"Found {len(noise_paths)} noise files")

print("\nCreating datasets...")
train_base = OptimizedMashupDataset(
    data_path, genres, stems, noise_paths,
    num_samples_per_genre=samples_per_genre_train
)

val_base = OptimizedMashupDataset(
    data_path, genres, stems, noise_paths,
    num_samples_per_genre=samples_per_genre_val
)

Found 2000 noise files

Creating datasets...


In [18]:
train_dataset = CNNDataset(train_base, augment=True)
val_dataset = CNNDataset(val_base, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Steps per epoch: {len(train_loader)}")

Train samples: 1500, Val samples: 300
Steps per epoch: 47


In [19]:
model = LightCNN(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=learning_rate,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs,
    pct_start=0.1
)

Total parameters: 102,026


In [20]:
print("\nStarting training...")
best_val_f1 = 0.0

for epoch in range(num_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*60}")

    train_loss, train_f1, train_acc = train_cnn_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )

    val_loss, val_f1, val_acc = validate_cnn(
        model, val_loader, criterion, device
    )

    print(f"\nTrain Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}, Val Acc: {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_f1": train_f1,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_f1": val_f1,
        "val_acc": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
        }, "cnn_best.pth")
        print(f"✓ Saved best CNN model (val_f1: {val_f1:.4f})")

wandb.finish()

print(f"\n{'='*60}")
print(f"✓ CNN Training Complete!")
print(f"Best Validation F1: {best_val_f1:.4f}")
print(f"{'='*60}")


Starting training...

Epoch 1/8


Training CNN:   0%|          | 0/47 [00:00<?, ?it/s]

Validating CNN:   0%|          | 0/10 [00:00<?, ?it/s]


Train Loss: 2.2048, Train F1: 0.1628, Train Acc: 17.40%
Val Loss: 2.1222, Val F1: 0.1193, Val Acc: 20.00%
✓ Saved best CNN model (val_f1: 0.1193)

Epoch 2/8


Training CNN:   0%|          | 0/47 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
self._shutdown_workers()
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
       if w.is_alive():
      Exception ignored in:   <function _MultiProcessingDataLoaderIter.__del__ at 0x7a8f7b158e00>^  ^ 
^^Traceback (most recent call last):
^^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloa

Validating CNN:   0%|          | 0/10 [00:00<?, ?it/s]


Train Loss: 1.5855, Train F1: 0.3821, Train Acc: 39.87%
Val Loss: 1.3932, Val F1: 0.3934, Val Acc: 44.00%


wandb: 
wandb: Run history:
wandb:         epoch ▁▂▃▄▅▆▇█
wandb: learning_rate ██▆▅▄▂▁▁
wandb:     train_acc ▁▅▆▇▇▇██
wandb:      train_f1 ▁▄▆▇▇▇██
wandb:    train_loss █▄▃▂▂▂▁▁
wandb:       val_acc ▁▂▅▆▇██▇
wandb:        val_f1 ▁▁▅▆▇██▇
wandb:      val_loss ██▄▃▂▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 8
wandb: learning_rate 0.0
wandb:     train_acc 39.86667
wandb:      train_f1 0.38215
wandb:    train_loss 1.58552
wandb:       val_acc 44
wandb:        val_f1 0.39343
wandb:      val_loss 1.39317
wandb: 
wandb: 🚀 View run cnn-from-scratch-optimized at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/0u8z9x96
wandb: ⭐️ View project at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260404_024027-0u8z9x96/logs



✓ CNN Training Complete!
Best Validation F1: 0.4256


In [21]:
print("\n" + "="*60)
print("CNN INFERENCE STARTING")
print("="*60)

cnn_model_inference = LightCNN(num_classes=num_classes).to(device)
cnn_checkpoint = torch.load("cnn_best.pth", map_location=device)
cnn_model_inference.load_state_dict(cnn_checkpoint['model_state_dict'])
cnn_model_inference.eval()
print(f"✓ Best CNN model loaded (F1: {cnn_checkpoint['val_f1']:.4f})")

test_df = pd.read_csv(data_path + "/test.csv")
print(f"✓ Found {len(test_df)} test files")

cnn_results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="CNN Predictions"):
    file_id = row["id"]
    file_path = data_path + "/" + row["filename"]

    audio = load_audio(file_path, sample_rate)

    if audio is None:
        predicted_genre = "pop"
    else:
        audio = crop_audio(audio, crop_duration, sample_rate)

        mel_spec = librosa.feature.melspectrogram(
            y=audio, sr=sample_rate, n_mels=n_mels,
            n_fft=n_fft, hop_length=hop_length
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        mel_spec_tensor = torch.FloatTensor(mel_spec_db).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            output = cnn_model_inference(mel_spec_tensor)
            probs = F.softmax(output, dim=1).cpu().numpy()[0]
            pred_idx = np.argmax(probs)
            predicted_genre = idx_to_genre[pred_idx]

    cnn_results.append({"id": file_id, "genre": predicted_genre})

cnn_submission = pd.DataFrame(cnn_results)
cnn_submission = cnn_submission.sort_values("id").reset_index(drop=True)
cnn_submission.to_csv("submission_cnn.csv", index=False)

print(f"\n✓ CNN submission created: submission_cnn.csv")


CNN INFERENCE STARTING
✓ Best CNN model loaded (F1: 0.4256)
✓ Found 3020 test files


CNN Predictions:   0%|          | 0/3020 [00:00<?, ?it/s]


✓ CNN submission created: submission_cnn.csv


# Model 3

In [22]:
dataset_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
audio_sr = 16000
clip_duration = 30
noise_snr_min = 5
noise_snr_max = 20

mel_bins = 64
fft_window = 1024
hop_stride = 512

audio_stems = ["drums", "vocals", "bass", "other"]
music_genres = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]

genre_to_label = {genre: idx for idx, genre in enumerate(music_genres)}
label_to_genre = {idx: genre for genre, idx in genre_to_label.items()}
num_output_classes = 10

train_batch_size = 24
total_epochs = 10
init_learning_rate = 1e-3
loader_workers = 4

train_samples_per_genre = 150
val_samples_per_genre = 30

compute_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {compute_device}")

Using device: cuda


In [23]:
def read_waveform(file_loc, sr=audio_sr):
    try:
        waveform, _ = librosa.load(file_loc, sr=sr, mono=True)
        return waveform
    except Exception as err:
        print(f"Error loading: {file_loc}")
        return None

def adjust_length(waveform, clip_sec, sr=audio_sr):
    target_len = clip_sec * sr

    if len(waveform) < target_len:
        waveform = np.pad(waveform, (0, target_len - len(waveform)))
    else:
        start_idx = random.randint(0, len(waveform) - target_len)
        waveform = waveform[start_idx:start_idx + target_len]

    return waveform

def inject_noise(waveform, noise_wave, snr_value):
    if len(noise_wave) < len(waveform):
        noise_wave = np.tile(noise_wave, int(np.ceil(len(waveform) / len(noise_wave))))
    noise_wave = noise_wave[:len(waveform)]

    start_idx = random.randint(0, len(waveform) // 2)
    end_idx = min(len(waveform), start_idx + len(noise_wave) // 2)

    signal_chunk = waveform[start_idx:end_idx]
    noise_chunk = noise_wave[:end_idx - start_idx]

    sig_power = np.mean(signal_chunk ** 2)
    noise_power = np.mean(noise_chunk ** 2)

    if noise_power > 0:
        snr_linear = 10 ** (snr_value / 10)
        scale_factor = np.sqrt(sig_power / (noise_power * snr_linear))
        waveform[start_idx:end_idx] += noise_chunk * scale_factor

    waveform = waveform / (np.max(np.abs(waveform)) + 1e-8)
    return waveform

def collect_noise_files(dataset_dir):
    noise_dir = Path(dataset_dir) / "ESC-50-master" / "audio"
    if not noise_dir.exists():
        return []
    return [str(file_path) for file_path in noise_dir.glob("*.wav")]

In [24]:
class MashupSequenceDataset(Dataset):

    def __init__(self, dataset_dir, genre_list, stem_types, noise_files,
                 samples_per_class=150, sr=audio_sr,
                 clip_sec=clip_duration):

        self.dataset_dir = Path(dataset_dir)
        self.genre_list = genre_list
        self.stem_types = stem_types
        self.noise_files = noise_files
        self.samples_per_class = samples_per_class
        self.sr = sr
        self.clip_sec = clip_sec

        self.audio_map = self._index_audio_files()
        self.dataset_size = len(genre_list) * samples_per_class

    def _index_audio_files(self):
        audio_map = {}
        for genre in self.genre_list:
            genre_dir = self.dataset_dir / "genres_stems" / genre
            audio_map[genre] = {stem: [] for stem in self.stem_types}

            if not genre_dir.exists():
                continue

            for track in os.listdir(genre_dir):
                track_dir = genre_dir / track
                if not track_dir.is_dir():
                    continue

                for stem in self.stem_types:
                    track_file = track_dir / f"{stem}.wav"
                    if track_file.exists():
                        audio_map[genre][stem].append(str(track_file))

        return audio_map

    def _create_sample(self, genre):

        chosen_files = []
        for stem in self.stem_types:
            if self.audio_map[genre][stem]:
                chosen_files.append(random.choice(self.audio_map[genre][stem]))

        if len(chosen_files) < 2:
            return None

        waveform_list = []
        for file_loc in chosen_files:
            waveform = read_waveform(file_loc, self.sr)
            if waveform is not None:
                waveform_list.append(waveform)

        if len(waveform_list) < 2:
            return None

        aligned_audio = waveform_list

        min_len = min(len(wave) for wave in aligned_audio)
        combined_wave = np.zeros(min_len)

        for wave in aligned_audio:
            weight = random.uniform(0.6, 1.4)
            combined_wave += weight * wave[:min_len]

        combined_wave = combined_wave / (np.max(np.abs(combined_wave)) + 1e-8)

        combined_wave = adjust_length(combined_wave, self.clip_sec, self.sr)

        if self.noise_files and random.random() > 0.2:
            noise_wave = read_waveform(random.choice(self.noise_files), self.sr)
            if noise_wave is not None:
                noise_wave = adjust_length(noise_wave, self.clip_sec, self.sr)
                snr_value = random.uniform(noise_snr_min, noise_snr_max)
                combined_wave = inject_noise(combined_wave, noise_wave, snr_value)

        return combined_wave

    def __len__(self):
        return self.dataset_size

    def __getitem__(self, index):

        genre_idx = index // self.samples_per_class
        genre = self.genre_list[genre_idx]

        waveform = None
        max_trials = 3
        for _ in range(max_trials):
            waveform = self._create_sample(genre)
            if waveform is not None:
                break

        if waveform is None:
            waveform = np.zeros(self.clip_sec * self.sr)

        label = genre_to_label[genre]

        return waveform, label

In [25]:
class SequenceFeatureDataset(Dataset):

    def __init__(self, source_dataset, use_augment=False):
        self.source_dataset = source_dataset
        self.use_augment = use_augment

        if use_augment:
            self.freq_aug = T.FrequencyMasking(freq_mask_param=8)
            self.time_aug = T.TimeMasking(time_mask_param=16)

    def __len__(self):
        return len(self.source_dataset)

    def __getitem__(self, index):
        waveform, target = self.source_dataset[index]

        spec = librosa.feature.melspectrogram(
            y=waveform,
            sr=audio_sr,
            n_mels=mel_bins,
            n_fft=fft_window,
            hop_length=hop_stride
        )

        spec_db = librosa.power_to_db(spec, ref=np.max)
        spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)

        spec_tensor = torch.FloatTensor(spec_db).unsqueeze(0)

        if self.use_augment:
            spec_tensor = self.freq_aug(spec_tensor)
            spec_tensor = self.time_aug(spec_tensor)

        return spec_tensor, target

In [26]:
class HybridAudioNet(nn.Module):

    def __init__(self, output_classes=10):
        super(HybridAudioNet, self).__init__()

        self.conv_block1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(32)
        self.downsample1 = nn.MaxPool2d(2, 2)

        self.conv_block2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm2d(64)
        self.downsample2 = nn.MaxPool2d(2, 2)

        self.conv_block3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.norm3 = nn.BatchNorm2d(128)
        self.downsample3 = nn.MaxPool2d(2, 2)

        self.sequence_model = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dense1 = nn.Linear(64 * 2, 64)
        self.regularizer = nn.Dropout(0.4)
        self.output_layer = nn.Linear(64, output_classes)

    def forward(self, x):

        x = self.downsample1(F.relu(self.norm1(self.conv_block1(x))))
        x = self.downsample2(F.relu(self.norm2(self.conv_block2(x))))
        x = self.downsample3(F.relu(self.norm3(self.conv_block3(x))))

        batch_size, channel_dim, freq_dim, time_dim = x.size()

        x = x.mean(dim=2)

        x = x.permute(0, 2, 1)

        x, _ = self.sequence_model(x)

        x = x.mean(dim=1)

        x = F.relu(self.dense1(x))
        x = self.regularizer(x)
        x = self.output_layer(x)

        return x

In [27]:
def run_training_epoch(network, data_stream, loss_fn, optim_fn, lr_scheduler, compute_device):
    network.train()
    total_loss = 0.0
    pred_buffer = []
    label_buffer = []

    progress_bar = tqdm(data_stream, desc="Training CRNN")

    for batch_inputs, batch_labels in progress_bar:
        batch_inputs = batch_inputs.to(compute_device)
        batch_labels = batch_labels.to(compute_device)

        optim_fn.zero_grad()

        predictions = network(batch_inputs)
        loss_value = loss_fn(predictions, batch_labels)

        loss_value.backward()

        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)

        optim_fn.step()
        lr_scheduler.step()

        total_loss += loss_value.item()

        pred_classes = predictions.argmax(dim=1).cpu().numpy()
        pred_buffer.extend(pred_classes)
        label_buffer.extend(batch_labels.cpu().numpy())

        accuracy = 100. * sum(p == l for p, l in zip(pred_buffer, label_buffer)) / len(label_buffer)
        progress_bar.set_postfix({'loss': f'{loss_value.item():.4f}', 'acc': f'{accuracy:.2f}%'})

    avg_loss = total_loss / len(data_stream)
    macro_f1 = f1_score(label_buffer, pred_buffer, average='macro')
    avg_acc = 100. * sum(p == l for p, l in zip(pred_buffer, label_buffer)) / len(label_buffer)

    return avg_loss, macro_f1, avg_acc


def run_validation_epoch(network, data_stream, loss_fn, compute_device):
    network.eval()
    total_loss = 0.0
    pred_buffer = []
    label_buffer = []

    with torch.no_grad():
        for batch_inputs, batch_labels in tqdm(data_stream, desc="Validating CRNN"):
            batch_inputs = batch_inputs.to(compute_device)
            batch_labels = batch_labels.to(compute_device)

            predictions = network(batch_inputs)
            loss_value = loss_fn(predictions, batch_labels)

            total_loss += loss_value.item()

            pred_classes = predictions.argmax(dim=1).cpu().numpy()
            pred_buffer.extend(pred_classes)
            label_buffer.extend(batch_labels.cpu().numpy())

    val_loss = total_loss / len(data_stream)
    val_f1 = f1_score(label_buffer, pred_buffer, average='macro')
    val_acc = 100. * sum(p == l for p, l in zip(pred_buffer, label_buffer)) / len(label_buffer)

    return val_loss, val_f1, val_acc

In [28]:
print("\n" + "="*60)
print("OPTIMIZED CRNN TRAINING ")
print("="*60)

os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="crnn-optimized",
    config={
        "model": "CRNN",
        "architecture": "CNN+LSTM",
        "batch_size": train_batch_size,
        "epochs": total_epochs,
        "learning_rate": init_learning_rate,
        "n_mels": mel_bins,
        "samples_per_genre": train_samples_per_genre,
        "lstm_hidden": 64,
        "lstm_layers": 1
    }
)

noise_files = collect_noise_files(dataset_dir)
print(f"Found {len(noise_files)} noise files")


OPTIMIZED CRNN TRAINING 


wandb: setting up run ssya9ooi
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260404_033413-ssya9ooi
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run crnn-optimized
wandb: ⭐️ View project at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: 🚀 View run at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/ssya9ooi


Found 2000 noise files


In [29]:
print("\nCreating datasets...")

train_core_dataset = MashupSequenceDataset(
    dataset_dir, music_genres, audio_stems, noise_files,
    samples_per_class=train_samples_per_genre
)

val_core_dataset = MashupSequenceDataset(
    dataset_dir, music_genres, audio_stems, noise_files,
    samples_per_class=val_samples_per_genre
)

train_feature_dataset = SequenceFeatureDataset(train_core_dataset, use_augment=True)
val_feature_dataset = SequenceFeatureDataset(val_core_dataset, use_augment=False)

train_data_stream = DataLoader(
    train_feature_dataset,
    batch_size=train_batch_size,
    shuffle=True,
    num_workers=loader_workers,
    pin_memory=True
)

val_data_stream = DataLoader(
    val_feature_dataset,
    batch_size=train_batch_size,
    shuffle=False,
    num_workers=loader_workers,
    pin_memory=True
)

print(f"Train samples: {len(train_feature_dataset)}, Val samples: {len(val_feature_dataset)}")
print(f"Steps per epoch: {len(train_data_stream)}")


Creating datasets...
Train samples: 1500, Val samples: 300
Steps per epoch: 63


In [30]:
audio_network = HybridAudioNet(output_classes=num_output_classes).to(compute_device)

param_count = sum(p.numel() for p in audio_network.parameters())
print(f"Total parameters: {param_count:,}")

loss_function = nn.CrossEntropyLoss()
optimizer_fn = torch.optim.AdamW(audio_network.parameters(), lr=init_learning_rate, weight_decay=0.01)

lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_fn,
    max_lr=init_learning_rate,
    steps_per_epoch=len(train_data_stream),
    epochs=total_epochs,
    pct_start=0.1
)

print("\nStarting training...")
best_validation_f1 = 0.0

for epoch_idx in range(total_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch_idx+1}/{total_epochs}")
    print(f"{'='*60}")

    train_loss, train_f1, train_acc = run_training_epoch(
        audio_network, train_data_stream, loss_function, optimizer_fn, lr_scheduler, compute_device
    )

    val_loss, val_f1, val_acc = run_validation_epoch(
        audio_network, val_data_stream, loss_function, compute_device
    )

    print(f"\nTrain Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}, Val Acc: {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch_idx + 1,
        "train_loss": train_loss,
        "train_f1": train_f1,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_f1": val_f1,
        "val_acc": val_acc,
        "learning_rate": optimizer_fn.param_groups[0]['lr']
    })

    if val_f1 > best_validation_f1:
        best_validation_f1 = val_f1
        torch.save({
            'epoch': epoch_idx,
            'model_state_dict': audio_network.state_dict(),
            'optimizer_state_dict': optimizer_fn.state_dict(),
            'val_f1': val_f1,
        }, "crnn_best.pth")
        print(f"✓ Saved best CRNN model (val_f1: {val_f1:.4f})")

wandb.finish()

print(f"\n{'='*60}")
print(f"✓ CRNN Training Complete!")
print(f"Best Validation F1: {best_validation_f1:.4f}")
print(f"{'='*60}")

Total parameters: 201,354

Starting training...

Epoch 1/10


Training CRNN:   0%|          | 0/63 [00:00<?, ?it/s]

Validating CRNN:   0%|          | 0/13 [00:00<?, ?it/s]


Train Loss: 2.1701, Train F1: 0.1783, Train Acc: 19.27%
Val Loss: 2.0171, Val F1: 0.1825, Val Acc: 21.67%
✓ Saved best CRNN model (val_f1: 0.1825)

Epoch 2/10


Training CRNN:   0%|          | 0/63 [00:00<?, ?it/s]

...

Training CRNN:   0%|          | 0/63 [00:00<?, ?it/s]

Validating CRNN:   0%|          | 0/13 [00:00<?, ?it/s]


Train Loss: 1.0973, Train F1: 0.5792, Train Acc: 58.60%
Val Loss: 0.9155, Val F1: 0.6898, Val Acc: 70.33%
✓ Saved best CRNN model (val_f1: 0.6898)


wandb: 
wandb: Run history:
wandb:         epoch ▁▂▃▃▄▅▆▆▇█
wandb: learning_rate ██▇▆▅▄▃▂▁▁
wandb:     train_acc ▁▄▅▅▆▇▇███
wandb:      train_f1 ▁▄▅▅▆▇▇███
wandb:    train_loss █▅▄▄▃▂▂▁▁▁
wandb:       val_acc ▁▁▄▄▆▆▆▇▆█
wandb:        val_f1 ▂▁▄▄▆▆▆▇▇█
wandb:      val_loss ██▄▄▃▃▃▂▂▁
wandb: 
wandb: Run summary:
wandb:         epoch 10
wandb: learning_rate 0.0
wandb:     train_acc 58.6
wandb:      train_f1 0.57919
wandb:    train_loss 1.09732
wandb:       val_acc 70.33333
wandb:        val_f1 0.68975
wandb:      val_loss 0.91545
wandb: 
wandb: 🚀 View run crnn-optimized at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/ssya9ooi
wandb: ⭐️ View project at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260404_033413-ssya9ooi/logs



✓ CRNN Training Complete!
Best Validation F1: 0.6898


In [31]:
print("\n" + "="*60)
print("CRNN INFERENCE STARTING")
print("="*60)

inference_model = HybridAudioNet(output_classes=num_output_classes).to(compute_device)
checkpoint_data = torch.load("crnn_best.pth", map_location=compute_device)
inference_model.load_state_dict(checkpoint_data['model_state_dict'])
inference_model.eval()
print(f"✓ Best CRNN model loaded (F1: {checkpoint_data['val_f1']:.4f})")

if 'test_df' not in locals():
    test_df = pd.read_csv(dataset_dir + "/test.csv")
    print(f"✓ Found {len(test_df)} test files")

prediction_records = []

for _, sample in tqdm(test_df.iterrows(), total=len(test_df), desc="CRNN Predictions"):
    sample_id = sample["id"]
    audio_file = dataset_dir + "/" + sample["filename"]

    waveform = read_waveform(audio_file, audio_sr)

    if waveform is None:
        predicted_label = "pop"
    else:
        waveform = adjust_length(waveform, clip_duration, audio_sr)

        spec = librosa.feature.melspectrogram(
            y=waveform, sr=audio_sr, n_mels=mel_bins,
            n_fft=fft_window, hop_length=hop_stride
        )
        spec_db = librosa.power_to_db(spec, ref=np.max)
        spec_db = (spec_db - spec_db.min()) / (spec_db.max() - spec_db.min() + 1e-8)
        spec_tensor = torch.FloatTensor(spec_db).unsqueeze(0).unsqueeze(0).to(compute_device)

        with torch.no_grad():
            logits = inference_model(spec_tensor)
            probabilities = F.softmax(logits, dim=1).cpu().numpy()[0]
            pred_index = np.argmax(probabilities)
            predicted_label = label_to_genre[pred_index]

    prediction_records.append({"id": sample_id, "genre": predicted_label})

submission_df = pd.DataFrame(prediction_records)
submission_df = submission_df.sort_values("id").reset_index(drop=True)
submission_df.to_csv("submission_crnn.csv", index=False)

print(f"\n✓ CRNN submission created: submission_crnn.csv")
print(f"Total predictions: {len(submission_df)}")



CRNN INFERENCE STARTING
✓ Best CRNN model loaded (F1: 0.6898)


CRNN Predictions:   0%|          | 0/3020 [00:00<?, ?it/s]


✓ CRNN submission created: submission_crnn.csv
Total predictions: 3020
